# 턱걸이 솔루션

<img src="hrnet_pose.png" width = "130px" height="200px" alt="error"></img>

기본 영상 이미지 : 720x1280

In [140]:
import numpy as np

In [141]:
img_size = [720, 1280]

In [142]:
# HRNET을 거친 좌표 예시
# 1 Frame 기준
coordinates = np.array([[[3.3067252e+02, 4.2102542e+02, 9.6591783e-01],
                        [3.4670767e+02, 4.1568036e+02, 9.6552366e-01],
                        [3.1998242e+02, 4.1568036e+02, 9.6760297e-01],
                        [3.7343295e+02, 4.2637045e+02, 9.6192151e-01],
                        [3.0929233e+02, 4.3706058e+02, 9.3673599e-01],
                        [3.8946811e+02, 4.7982098e+02, 9.3456656e-01],
                        [2.8791211e+02, 4.7982098e+02, 9.3135309e-01],
                        [4.2688345e+02, 3.8361005e+02, 9.5472604e-01],
                        [2.3446159e+02, 3.9964520e+02, 9.8633683e-01],
                        [4.4291861e+02, 2.8739911e+02, 9.7719079e-01],
                        [2.1308139e+02, 3.0343427e+02, 9.7313571e-01],
                        [3.7877798e+02, 6.6689777e+02, 9.0073270e-01],
                        [3.0929233e+02, 6.7758789e+02, 8.9935786e-01],
                        [3.8412305e+02, 8.2190430e+02, 9.1421479e-01],
                        [3.0394727e+02, 8.2190430e+02, 9.4648373e-01],
                        [3.6274283e+02, 9.5553058e+02, 9.4261879e-01],
                        [3.1463736e+02, 9.5018555e+02, 9.1504067e-01]]])

In [143]:
print(coordinates.shape)

(1, 17, 3)


In [144]:
coordinates = coordinates[0]

In [145]:
print(coordinates.shape)

(17, 3)


In [146]:
IsArmStretch = 0 # 팔은 쭉 폈는가
IsArmClose = 0 # 팔이 잘 벌려져 있는가

IsLegStretch = 0 # 다리를 일자로 쭉 폈는가
IsLegClose = 0 # 다리를 벌리지 않았는가

IsKneeStop = 1 # 무릎을 굽히지 않았는가
IsChinOver = 0 # 턱이 봉을 넘었는가

## 1. 상체

### 1.1. 팔이 펴진 각도 (어깨 - 팔꿈치 - 손목)

왼팔 : 5-7-9  
오른팔 : 6-8-10

In [147]:
# 넘어야 함
arm_stretch_angle = int(input('팔이 펴진 각도를 입력해 주세요 : ')) # ex. 130

In [148]:
def calculate_angle2D_3point(a, b, c, direction=-1):
    """
    calculate_angle2D is divided by left and right side because this function uses external product
    input : a,b,c -> landmarks with shape [x,y,z,visibility]
          direction -> int -1 or 1
                      -1 means Video(photo) for a person's left side and 1 means Video(photo) for a person's right side
    output : angle between vector ba and bc with range 0~360
    """
    # external product's z value
    external_z = (b[0]-a[0])*(b[1]-c[1]) - (b[1]-a[1])*(b[0]-c[0])

    a = np.array(a[:2]) #first
    b = np.array(b[:2]) #mid
    c = np.array(c[:2]) #end

    ba = b-a
    bc = b-c
    dot_result = np.dot(ba, bc)


    ba_size = np.linalg.norm(ba)
    bc_size = np.linalg.norm(bc)
    radi = np.arccos(dot_result / (ba_size*bc_size))
    angle = np.abs(radi*180.0/np.pi)

    
    if external_z * direction > 0:
        angle = 360 - angle
    
    return round(angle, 2)

In [149]:
# 팔 관절값
joint_idx = {'left_arm':[5, 7, 9], 'right_arm':[6,8,10]}

In [150]:
for k, v in joint_idx.items():
    first = coordinates[v[0]]
    mid = coordinates[v[1]]
    end = coordinates[v[2]]
    
    if 'right' in k:
        dimension = 1 # < 각도
    else:
        dimension = -1 # > 각도
    
    angle_tmp = calculate_angle2D_3point(first ,mid, end, dimension)
    print(angle_tmp)
    
    # 각도가 지정한 각도를 넘었다면
    if angle_tmp >= arm_stretch_angle:
        IsArmStretch = 1
    else:
        IsArmStretch = 0
        break

168.21
158.84


In [151]:
IsArmStretch

1

### 1.2. 두 팔 사이의 각도 (왼쪽 손목 - 목 아래 - 오른쪽 손목)

9-(5,6 중점)-10

In [152]:
# 몇 도 안쪽으로 들어와야 하는지
arm_between_angle = int(input('두 팔 사이의 각도를 입력해 주세요 : ')) # 80

In [153]:
def calculate_angle2D_4point(a, b1, b2, c, direction=-1):
    """
    calculate_angle2D is divided by left and right side because this function uses external product
    input : a,b,c -> landmarks with shape [x,y,z,visibility]
          direction -> int -1 or 1
                      -1 means Video(photo) for a person's left side and 1 means Video(photo) for a person's right side
    output : angle between vector ba and bc with range 0~360
    """
    b1 = np.array(b1[:2])
    b2 = np.array(b2[:2])
    
    b = [round(abs((b1[0]+b2[0])/2),2), round(abs((b1[1]+b2[1])/2),2)]
    
    # external product's z value
    external_z = (b[0]-a[0])*(b[1]-c[1]) - (b[1]-a[1])*(b[0]-c[0])

    a = np.array(a[:2]) #first
    b = np.array(b[:2]) #mid
    c = np.array(c[:2]) #end

    
    ba = b-a
    bc = b-c
    dot_result = np.dot(ba, bc)

    ba_size = np.linalg.norm(ba)
    bc_size = np.linalg.norm(bc)
    radi = np.arccos(dot_result / (ba_size*bc_size))
    angle = np.abs(radi*180.0/np.pi) # 60분법 변환

    
    if external_z * direction > 0:
        angle = 360 - angle
    
    return round(angle,2)

In [154]:
# 왼 손목 - 목 - 오른 손목
joint_idx = {'arm_between':[9, 5, 6, 10]}

In [155]:
for k, v in joint_idx.items():
    first = coordinates[v[0]]
    mid1 = coordinates[v[1]]
    mid2 = coordinates[v[2]]
    end = coordinates[v[3]]
   
    angle_tmp = calculate_angle2D_4point(first ,mid1, mid2, end, 1)
    print(angle_tmp)
    # 각도가 지정한 각도를 넘지 않았다면
    if angle_tmp <= arm_between_angle:
        IsArmClose = 1
    else:
        IsArmClose = 0
        break

63.9


In [156]:
IsArmClose

1

## 2. 하체

### 2.1. 다리가 펴진 각도 (고관절 - 무릎 - 발목)
왼다리 : 11-13-15  
오른다리 : 12-14-16

In [157]:
# 몇 도이상 펴져야 하는지
leg_stretch_angle = int(input('다리가 펴진 각도를 입력해 주세요 : ')) # 150

In [158]:
# 다리 관절값
joint_idx = {'left_leg':[11, 13, 15], 'right_leg':[12, 14, 16]}

In [159]:
for k, v in joint_idx.items():
    first = coordinates[v[0]]
    mid = coordinates[v[1]]
    end = coordinates[v[2]]
    
    if 'left' in k:
        dimension = 1 # < 각도
    else:
        dimension = -1 # > 각도
    
    angle_tmp = calculate_angle2D_3point(first ,mid, end, dimension)
    print(angle_tmp)
    
    # 각도가 지정한 각도를 넘었다면
    if angle_tmp >= leg_stretch_angle:
        IsLegStretch = 1
    else:
        IsLegStretch = 0
        break

168.93
173.12


In [160]:
IsLegStretch

1

### 2.2. 다리 사이의 각도 (왼쪽 무릎 - 허리중심 - 오른쪽 무릎)
13-(11, 12 중점)-14

언제 측정할 건데? : 계속  
가장 낮은 발의 y축 좌표가 이전 2번의 프레임보다 모두 높아졌을 때 기준 n번 이상 경고가 나오면 IsLegClose를 0으로

In [161]:
# 넘으면 안됨
leg_between_angle = int(input('다리 사이의 각도를 입력해 주세요 : ')) # 60

In [162]:
# 다리 관절값
joint_idx = {'leg_between':[13, 11, 12, 14]} 

In [163]:
for k, v in joint_idx.items():
    first = coordinates[v[0]]
    mid1 = coordinates[v[1]]
    mid2 = coordinates[v[2]]
    end = coordinates[v[3]]
   
    angle_tmp = calculate_angle2D_4point(first ,mid1, mid2, end, -1)
    print(angle_tmp)
    # 각도가 지정한 각도를 넘지 않았다면
    if angle_tmp <= leg_between_angle:
        IsLegClose = 1
    else:
        IsLegClose = 0
        break

29.99


In [164]:
IsLegClose

1

### 2.3. 발목의 높이 (무릎 y값 - 발목 y값)
왼다리 : 13-15
오른다리 : 14-16

발목의 y값이 무릎의 y값보다 작아지면 안됨

In [165]:
# 다리 관절값
joint_idx = {'left_leg':[13, 15], 'right_leg':[14, 16]}

In [166]:
for k, v in joint_idx.items():
    knee = coordinates[v[0]][:2] # 무릎 좌표
    ankle = coordinates[v[1]][:2]
    print(knee, ankle)
    
    knee_y = knee[1]
    ankle_y = ankle[1]
    
    if ankle_y <= knee_y:
        IsKneeStop = 0
        break

[384.12305 821.9043 ] [362.74283 955.53058]
[303.94727 821.9043 ] [314.63736 950.18555]


In [167]:
IsKneeStop # 무릎을 굽히지 않았는가

1

## 3. 턱이 봉을 넘었을 때

### 3.1. 예측 봉 위치
왼쪽 손목 : 9  
오른쪽 손목 : 10

In [168]:
# 우선 손목의 위치로 판단
# 작은 값이 더 위에 있는 것
left_wrist = coordinates[9][:2][1]
right_wrist = coordinates[10][:2][1]

In [169]:
bar_location = min(left_wrist,right_wrist);bar_location

287.39911

### 3.2. 예측 턱 위치
귀\~코 : 코\~턱 = 0.5 : 0.8  
코\~턱  = 1.6 \* 귀\~코

In [170]:
joint_idx = {'left_ear_nose':[3, 0], 'right_ear_nose':[4,0]} 

In [171]:
def calculate_chin(a, b):
    chin_x = b[0]
    a = np.array(a[1]) # ear,  y 좌표
    b = np.array(b[1]) # nose, y 좌표
    
    ear_to_nose = abs(a-b)
    # x좌표는 nose, y좌표는 nose + 
    chin = [chin_x, b+round(ear_to_nose*1.6,2)]
    
    return chin

In [172]:
expected_chin = []
for k, v in joint_idx.items():
    ear = coordinates[v[0]]
    nose = coordinates[v[1]]
    expected_chin.append(calculate_chin(ear ,nose))

chin_location = min(expected_chin)[1];chin_location

429.57542

In [173]:
if chin_location <= bar_location:
    IsChinOver = 1
else:
    IsChinOver = 0

In [174]:
IsChinOver

0

In [175]:
result = [IsArmStretch, IsArmClose, IsLegStretch, IsLegClose, IsKneeStop, IsChinOver];result

[1, 1, 1, 1, 1, 0]

In [176]:
import pandas as pd

In [182]:
aaaaa = pd.DataFrame({'IsArmStretch' : [0], 
              'IsArmClose' : [0],
              'IsLegStretch' : [0],
              'IsLegClose' : [0],
              'IsKneeStop' : [0],
              'IsChinOver' : [0],
              'Count':[0]})

In [187]:
aaaaa2 = pd.DataFrame({'IsArmStretch' : [0], 
              'IsArmClose' : [0],
              'IsLegStretch' : [0],
              'IsLegClose' : [0],
              'IsKneeStop' : [0],
              'IsChinOver' : [0],
              'Count':[1]})

In [184]:
bbbbb = pd.DataFrame()

In [186]:
b2 = pd.concat([bbbbb, aaaaa]);b2

,IsArmStretch,IsArmClose,IsLegStretch,IsLegClose,IsKneeStop,IsChinOver,Count
0,0,0,0,0,0,0,0


In [190]:
pd.concat([b2, aaaaa2])

,IsArmStretch,IsArmClose,IsLegStretch,IsLegClose,IsKneeStop,IsChinOver,Count
0,0,0,0,0,0,0,0
0,0,0,0,0,0,0,1


조건 : 130, 80, 150, 60

### 해야할 일

1. 코드에서 좌표 값을 언제 저장할 건지?
-> 좌표 값은 매번 뽑혀서 나옴

2. 손을 떼고 떨어지면 끝!



In [191]:
# 결과 
pd.read_csv('/home/complexion/데이터바우처/pose-estimator/result.csv')

,IsArmStretch,IsArmClose,IsLegStretch,IsLegClose,IsKneeStop,IsChinOver
0,1,1,1,1,1,0
1,1,1,1,1,1,0
2,1,1,1,1,1,0
3,1,1,1,1,1,0
4,1,1,1,1,1,0
...,...,...,...,...,...,...
486,1,0,1,1,1,1
487,1,0,1,1,1,1
488,1,0,1,1,1,1
489,1,0,1,1,1,1
